**ДЗ. Подбор параметров сети**

---

**Программа Распознавание цифр (с dropout ).**

---

Задания не предполагают изменение архитектуры сети, работаем с заданной сетью (полносвязная, слои заданы, количество нейронов можно менять) - программа, которая была в классе, выложена "Распознавание цифр с drop"

1. Поэкспериментируйте с подбором параметров, получите оптимальную сеть по точности распознавания, времени обучения, количеству примеров и т.п. Для экспериментов определите (и напишите, как Вы определили) критерии качества работы сети, критерии оптимальности:

    каков должен быть размер обучающего набора? Какое число нейронов  во внутреннем слое для него оптимально? Сколько в разных случаях надо эпох для обучения -  и прочие параметры.

2. Исследовать различные методы оптимизации, реализованные в keras (документация и пояснения про оптимизацию выложены).

Необходимо выбрать не менее 3 методов оптимизации из списка: SGD, RMSProp, Adadelta, Adam, Adagrad, Adamax, AdamW, Ftrl, Nadam
Для каждого выбранного метода оптимизации необходимо:
Подобрать оптимальные параметры с помощью кросс-валидации
Обучить полносвязную нейронную сеть с подобранными параметрами на обучающей выборке
Оценить качество нейронной сети на обучающей и тестовой выборке
Составить таблицу, сравнивающую методы оптимизации: для каждого указать необходимое число эпох для обучения, качество на обучающей и тестовой выборке, подобранные параметры.
В качестве ответа приложить ноутбук со всеми проведенными экспериментами и текстовый отчет с описанием экспериментов, итоговой таблицей и выводами

---

In [ ]:
"""
Импорты
"""

import os
os.environ['TF_CPP_MIN_LOG_LEVEL']='2'
import time
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist
from tensorflow import keras
from tensorflow.keras.layers import Dense, Flatten, Dropout
import tensorflow as tf
import pandas as pd

In [ ]:
"""
Фиксация начальных данных
"""

np.random.seed(42)
tf.random.set_seed(42)

(x_train, y_train), (x_test, y_test) = mnist.load_data()

x_train = x_train / 255
x_test = x_test / 255

y_train_cat = keras.utils.to_categorical(y_train, 10)
y_test_cat = keras.utils.to_categorical(y_test, 10)

results_table = []

In [ ]:
def main(limit=4000, kn=10, drop=0.9, epochs=50, bsize=32, optimizer='adam'):
    """
    Подать:
    Размер обучающего набора
    Количество нейронов
    Dropout rate
    Количество эпох
    Batch size
    Оптимизатор
    """
    
    print(f"Вызов с: limit={limit}, neurons={kn}, dropout={drop}, epochs={epochs}, batch_size={bsize}, optimizer={optimizer}")

    start_time = time.time()
    
    x_train_data = x_train[:limit]
    y_train_data = y_train_cat[:limit]
    
    x_valid = x_train[limit:limit*2]
    y_valid = y_train_cat[limit:limit*2]
    
    model = keras.Sequential([
        Flatten(input_shape=(28, 28, 1)),
        Dense(kn, activation='relu'),
        Dropout(drop),
        Dense(10, activation='softmax')
    ])

    model.compile(optimizer=optimizer,
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    
    history = model.fit(x_train_data, y_train_data, epochs=epochs, batch_size=bsize, 
                       validation_data=(x_valid, y_valid), verbose=False)
    
    test_loss, test_accuracy = model.evaluate(x_test, y_test_cat, verbose=False)

    training_time = time.time() - start_time
    
    pred = model.predict(x_test, verbose=False)
    pred = np.argmax(pred, axis=1)
    errors = np.sum(pred != y_test)
    
    # Запись в таблицу
    result = {
        'limit': limit,
        'neurons': kn,
        'dropout': drop,
        'epochs': epochs,
        'batch_size': bsize,
        'optimizer': optimizer,
        'test_accuracy': test_accuracy,
        'test_loss': test_loss,
        'training_time': training_time,
        'errors': errors,
        'final_train_acc': history.history['accuracy'][-1],
        'final_val_acc': history.history['val_accuracy'][-1]
    }
    
    results_table.append(result)
    
    # Вывод результатов
    print(f"Результат: accuracy={test_accuracy:.4f}, errors={errors}, time={training_time:.2f}с")
    print()
    
    return test_accuracy, errors, training_time

In [ ]:
def intelligent_parameter_search(optimizer, max_iterations=20):
    
    parameters = {
        'names': ['limit', 'neurons', 'dropout', 'batch_size', 'epochs'],
        'min': [500, 10, 0.01, 8, 10],
        'max': [25000, 1000, 0.98, 128, 100],
        'current': [2000, 100, 0.5, 32, 30],
        'active': [True, True, True, True, True],
        'step_factors': [0.3, 0.3, 0.2, 0.4, 0.3]
    }
    
    best_accuracy = 0
best_params = parameters['current'].copy()

for iteration in range(max_iterations):
    print(f"Итерация {iteration + 1}: {parameters['current']}")
    
    current_accuracy, _, _ = main(
        limit=parameters['current'][0],
        kn=parameters['current'][1],
        drop=parameters['current'][2],
        epochs=parameters['current'][4],
        bsize=parameters['current'][3],
        optimizer=optimizer
    )
    
    if current_accuracy > best_accuracy + 0.001:
        best_accuracy = current_accuracy
        best_params = parameters['current'].copy()
    
    for i in range(len(parameters['current'])):
        if not parameters['active'][i]:
            continue
            
        step_size = (parameters['max'][i] - parameters['min'][i]) * parameters['step_factors'][i]
        
        new_value = min(parameters['current'][i] + step_size, parameters['max'][i])
        test_params = parameters['current'].copy()
        test_params[i] = new_value
        
        test_accuracy, _, _ = main(
            limit=test_params[0], kn=test_params[1], drop=test_params[2],
            epochs=test_params[4], bsize=test_params[3], optimizer=optimizer
        )
        
        if test_accuracy > current_accuracy + 0.002:
            parameters['current'][i] = new_value
            continue
        
        new_value = max(parameters['current'][i] - step_size, parameters['min'][i])
        test_params[i] = new_value
        
        test_accuracy, _, _ = main(
            limit=test_params[0], kn=test_params[1], drop=test_params[2],
            epochs=test_params[4], bsize=test_params[3], optimizer=optimizer
        )
        
        if test_accuracy > current_accuracy + 0.002:
            parameters['current'][i] = new_value
        else:
            parameters['step_factors'][i] *= 0.8
            if parameters['step_factors'][i] < 0.05:
                parameters['active'][i] = False

return best_params, best_accuracy

In [ ]:
def compare_optimizers(optimizers_list):
    
    print("Сравнение оптимизаторов")
    
    best_results = {}
    
    for optimizer in optimizers_list:
        print(f"\nОптимизатор: {optimizer}")
        
        best_params, best_accuracy = intelligent_parameter_search(optimizer)
        best_results[optimizer] = {
            'params': best_params,
            'accuracy': best_accuracy
        }
    
    print("\nИтоговая таблица результатов:")
    final_df = pd.DataFrame(results_table)
    print(final_df.to_string(index=False))
    
    final_df.to_csv('optimization_results.csv', index=False)
    print("Результаты сохранены в 'optimization_results.csv'")
    
    return best_results, final_df

if __name__ == "__main__":
    optimizers_to_test = ['adam', 'rmsprop', 'sgd']
    
    best_results, results_df = compare_optimizers(optimizers_to_test)
    
    print("\nЛучшие результаты:")
    for optimizer, result in best_results.items():
        print(f"{optimizer}: accuracy = {result['accuracy']:.4f}")